# Proyecto ML 2025-2026 — Documentos Desclasificados del 23-F

**Notebook orquestador** — toda la lógica vive en `src/`.

| Caso | Clase | Descripción |
|------|-------|-------------|
| 1 | `EDACorpus` | Radiografía del corpus (EDA + clustering TF-IDF) |
| 2 | `ActorGraphCaso` | Grafo de co-menciones de actores |
| 3 | `TopicModelCaso` | Topic modeling semántico (BERTopic / NMF) |
| 4 | `ClasificadorCaso` | Clasificador supervisado de ministerio |
| 5 | `SpatioTemporalCaso` | Serie temporal + anomalías + geografía |

## 0. Setup

In [1]:
import logging
import sys
from pathlib import Path

# Añadir raíz del proyecto al path para importar src/
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s — %(message)s",
    datefmt="%H:%M:%S",
)
print(f"Raíz del proyecto: {ROOT}")

Raíz del proyecto: /Users/alejandro/Documents/Máster/ProyectoML/machine_learning


## 1. Obtención de datos

In [2]:
from src.data.scraper import Scraper23F

scraper = Scraper23F(
    base_url="https://23fbuscador.rtve.es",
    raw_path="data/documentos_23f_raw.json",
    delay=0.4,
)
raw = scraper.scrape(force_refresh=False)
print(f"Documentos cargados: {len(raw)}")

11:42:54 [src.data.scraper] INFO — Obteniendo listado de documentos...
11:42:57 [src.data.scraper] INFO — 167 documentos en el listado
11:44:33 [src.data.scraper] INFO — Scraping completado: 167 documentos
11:44:33 [src.data.scraper] INFO — Datos guardados en data/documentos_23f_raw.json


Documentos cargados: 167


## 2. Construcción del corpus

In [3]:
from src.data.builder import CorpusBuilder

builder = CorpusBuilder(raw)
df = builder.build()

print(f"Shape: {df.shape}")
df[["id","titulo","fuente","anio","periodo","paginas",
    "n_personas","n_palabras_ocr","riqueza_lexica"]].head(5)

11:44:33 [src.data.builder] INFO — DataFrame construido: 167 docs × 19 variables


Shape: (167, 19)


,id,titulo,fuente,anio,periodo,paginas,n_personas,n_palabras_ocr,riqueza_lexica
0,1860,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,3,10,640,0.6198
1,1859,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,4,10,1018,0.4658
2,1858,Vista oral 2/81 del Consejo Supremo de Justici...,Interior,1982.0,Proceso judicial,5,10,1347,0.5258
3,1857,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,6,10,1826,0.4628
4,1856,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,6,10,1740,0.4328


In [4]:
# Vista rápida de la distribución
print(df["fuente"].value_counts().to_string())
print()
print(df["periodo"].value_counts().to_string())

fuente
Defensa           81
Interior          59
Exteriores        15
No determinado    12

periodo
Desconocido         78
Proceso judicial    64
23-F (1981)         21
Pre-golpe            3
Post-proceso         1


## 3. Caso 1 — EDA y Clustering del Corpus

In [ ]:
from src.casos.caso1_eda import EDACorpus

caso1 = EDACorpus(df, output_dir="outputs", fig_dir="figuras")
results1 = caso1.run()
caso1.export()
print(caso1.summary())